# Этап 0: mini-GPT на Kaggle

Перед запуском:
1. **Settings → Accelerator → GPU T4 x2** (или P100).
2. **Settings → Internet → On** (нужен телефон, подтверждённый в аккаунте Kaggle).
3. Если репозиторий приватный, сделайте его публичным или клонируйте по токену.
4. (Необязательно) W&B: **Add-ons → Secrets**, добавьте секрет `WANDB_API_KEY`.

Код живёт в git-репозитории, ноутбук только клонирует его и запускает скрипты.

In [ ]:
REPO_URL = "https://github.com/Pavel-V418/MyMiniLLM.git"
BRANCH = "master"
CORPUS_MB = 200        # сколько МБ Википедии скачать для обучения
CONFIG = "small"       # пресет из mini_llm/configs.py
USE_WANDB = False      # True, если добавили секрет WANDB_API_KEY

DATA = "/tmp/data"                       # данные не попадают в Output ноутбука
OUT = "/kaggle/working/checkpoints/stage0"  # чекпоинты попадут в Output

!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
if os.path.exists("MyMiniLLM"):
    !cd MyMiniLLM && git pull
else:
    !git clone -b {BRANCH} {REPO_URL} MyMiniLLM
%cd MyMiniLLM
!git log -1 --oneline
!pip install -q regex

In [ ]:
if USE_WANDB:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

## Данные: скачать корпус и токенизировать нашим BPE

In [ ]:
!python scripts/prepare_corpus.py --mb {CORPUS_MB} --out {DATA}/corpus
!python scripts/tokenize_corpus.py --corpus {DATA}/corpus --out {DATA}/tokens

## Обучение

Если сессия оборвалась, запустите ячейку снова с `--resume`: обучение продолжится с последнего чекпоинта.

In [ ]:
wandb_flag = "--wandb" if USE_WANDB else ""
!python scripts/train.py --config {CONFIG} --data-dir {DATA}/tokens --out-dir {OUT} {wandb_flag}

## Генерация

In [ ]:
!python scripts/generate.py --ckpt {OUT}/best.pt --prompt "Москва — это" --num-samples 3 --max-new-tokens 100
!python scripts/generate.py --ckpt {OUT}/best.pt --prompt "В 1812 году" --num-samples 2 --max-new-tokens 100

## График loss

`best.pt` и `metrics.jsonl` скачайте из **Output** справа.

In [ ]:
import json
import matplotlib.pyplot as plt

rows = [json.loads(l) for l in open(f"{OUT}/metrics.jsonl", encoding="utf-8")]
tr = [(r["step"], r["train/loss"]) for r in rows if "train/loss" in r]
va = [(r["step"], r["val/loss"]) for r in rows if "val/loss" in r]
plt.figure(figsize=(8, 4))
plt.plot(*zip(*tr), label="train", alpha=0.6)
plt.plot(*zip(*va), "o-", label="val")
plt.xlabel("шаг"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=0.3)
plt.savefig(f"{OUT}/loss.png", dpi=120, bbox_inches="tight")
plt.show()